# UCF101 as a real video stream (Colab)

In the TDA paper UCF101 is one image per video (the middle frame). Here every test video becomes a short clip
of **16 consecutive frames at 5 fps** (about 3 seconds around the middle), so TDA sees a real video stream.

The notebook downloads UCF101 (7 GB, on Colab's temporary disk), extracts CLIP features straight from the
`.avi` files (nothing is unzipped), saves only the feature files on Drive, then runs the stream experiments with
the same `streams.py` used for CORe50. UCF101 *groups* (`g01`-`g25`, same actor and background) play the role
of CORe50 sessions, so S0, S1, S2 and S3 are all defined.

**Pilot first** (`PILOT = True`): 3 videos per class (~300 videos, ~5,000 frames), about 30 minutes in total.
Check the numbers at the end, then set `PILOT = False` for the full test split (3,783 videos, ~60,000 frames).

*Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

In [ ]:
!nvidia-smi -L

## Settings

In [ ]:
# ======================= SETTINGS =======================
REPO_URL  = "https://github.com/gretadell/Visionaries-group-project---TDA.git"
PILOT     = False                         # True: 3 videos per class; False: full test split
BACKBONES = ["RN50", "ViT-B/16"]
ORDERS    = "s0,s1,s2,s3"
MODES     = "tda,clip,m2,oracle"
SEEDS     = 10
# ========================================================

import os
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/TDA_replication"
CODE = "/content/TDA_replication"
UCF = "/content/ucf101"                   # temporary disk: the 7 GB zip is not kept
NAME = "ucf101_video_pilot" if PILOT else "ucf101_video"
os.environ.update({"TDA_HOME": CODE, "REPO_DIR": f"{CODE}/TDA", "RESULTS_DIR": f"{DRIVE_ROOT}/results"})
os.makedirs(f"{DRIVE_ROOT}/results/features", exist_ok=True)
os.makedirs(UCF, exist_ok=True)

## Code

In [ ]:
if os.path.isdir(f"{CODE}/.git"):
    !git -C {CODE} fetch -q origin && git -C {CODE} reset -q --hard origin/main
else:
    !git clone -q {REPO_URL} {CODE}
if not os.path.isdir(f"{CODE}/TDA/.git"):
    !git clone -q https://github.com/kdiAAA/TDA.git {CODE}/TDA
!pip -q install ftfy regex opencv-python-headless
!git -C {CODE} log -1 --format="this repository: %h %cd %s"
%cd {CODE}

## Download UCF101 (about 5 minutes)
Videos and official train/test lists, from the Hugging Face copy of the official files.

In [ ]:
HF = "https://huggingface.co/datasets/quchenyuan/UCF101-ZIP/resolve/main"
for f in ["UCF101TrainTestSplits-RecognitionTask.zip", "UCF-101.zip"]:
    if not os.path.isfile(f"{UCF}/{f}"):
        !wget -q --show-progress -O {UCF}/{f} {HF}/{f}
!ls -lh {UCF}

## Extract CLIP features (pilot: a few minutes per backbone)
Feature files already on Drive are not recomputed.

In [ ]:
pilot = "--max-per-class 3" if PILOT else ""
for bb in BACKBONES:
    out = f"{DRIVE_ROOT}/results/features/{NAME}__{bb.replace('/', '-')}.pt"
    if os.path.isfile(out):
        print("already there:", out)
        continue
    !python extract_ucf101_video.py --video-zip {UCF}/UCF-101.zip --split-zip {UCF}/UCF101TrainTestSplits-RecognitionTask.zip --backbone {bb} --name {NAME} {pilot} --workers 2

## Stream runs
Hyper-parameters: the official UCF101 config of the paper (`--cfg-like ucf101`), so nothing is tuned on these videos.
Also the video baseline (CLIP smoothed over the last 5 / 15 frames) on the temporal orders.

In [ ]:
for bb in BACKBONES:
    !python streams.py run --dataset {NAME} --backbone {bb} --cfg-like ucf101 --order {ORDERS} --rho 0.5,0.9 --mode {MODES} --repeats {SEEDS} --trace
    !python streams.py run --dataset {NAME} --backbone {bb} --cfg-like ucf101 --order s3,instance --mode smooth --window 5,15 --repeats {SEEDS}

## Results and pilot checklist
Decide whether to run the full version by looking at three things:
1. **CLIP alone** (the `clip` rows) should be in the same range as CLIP on UCF101 in our replication (one middle frame per video). Much lower means the frames or prompts are wrong.
2. **TDA − CLIP** should be positive in S0, as in the paper.
3. **Do the orders differ?** Compare S0, S2, S3 with their ± intervals. With 5 seeds the intervals are wide; only clear differences matter here.

If 1 and 2 hold, set `PILOT = False` and run all again (about 1 hour more).

In [ ]:
!python streams.py summary --dataset {NAME}
for bb in BACKBONES:
    !python analyze_traces.py --dataset {NAME} --backbone {bb} --modes tda,clip,m2,oracle --config ucf101
print("plots in", f"{DRIVE_ROOT}/results/analysis/")